# PG Primary + Obfuscated ES Search

**Routing pattern:**
- WRITE: `[postgresql, elasticsearch_obfuscated]` — PG stores full items; ES obfuscated stores geoid tokens only
- READ: `[postgresql]` — browsing via PG
- SEARCH: `[elasticsearch_obfuscated]` — geoid-only anonymized search

Use this pattern when item data is sensitive but spatial discovery is needed without exposing attributes.

In [ ]:
import httpx

BASE = "http://localhost:8000"
CATALOG_ID = "demo-obfuscated"
COLLECTION_ID = "protected-parcels"

client = httpx.AsyncClient(base_url=BASE, timeout=30)
print("Client ready")

## Step 1 — Create catalog and collection

In [ ]:
r = await client.post(f"/stac/catalogs", json={"id": CATALOG_ID, "title": "Demo Obfuscated", "description": "Obfuscated ES demo"})
print(r.status_code, r.json() if r.status_code not in (200, 201, 409) else "OK")

In [ ]:
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Protected Parcels",
    "description": "Land parcels — attributes anonymized in search",
    "license": "proprietary",
})
print(r.status_code, r.json() if r.status_code not in (200, 201, 409) else "OK")

## Step 2 — Configure routing

In [ ]:
routing = {
    "plugin_id": "collection:drivers",
    "operations": {
        "WRITE": [
            {"driver_id": "DriverRecordsPostgresql", "on_failure": "fatal", "write_mode": "sync"},
            {"driver_id": "DriverRecordsElasticsearchObfuscated", "on_failure": "warn", "write_mode": "async"}
        ],
        "READ": [{"driver_id": "DriverRecordsPostgresql"}],
        "SEARCH": [{"driver_id": "DriverRecordsElasticsearchObfuscated"}],
        "METADATA": [{"driver_id": "DriverRecordsPostgresql"}]
    }
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/collection:drivers",
    json=routing,
)
print(r.status_code, r.text[:200])

## Step 3 — Insert test items

In [ ]:
import json

features = [
    {
        "type": "Feature",
        "id": f"parcel-{i:03d}",
        "geometry": {"type": "Point", "coordinates": [12.5 + i * 0.01, 41.9 + i * 0.01]},
        "properties": {"owner": f"Owner {i}", "area_m2": 500 + i * 10, "cadastral_ref": f"IT{i:05d}"}
    }
    for i in range(5)
]

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": features},
)
print(r.status_code, f"{len(features)} items sent")

## Step 4 — Read from PG (full attributes)

In [ ]:
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=10")
data = r.json()
print(f"Items returned: {data.get('numberReturned', len(data.get('features', [])))}")
for f in data.get("features", []):
    print(" ", f["id"], f["properties"].get("owner"))

## Step 5 — Search via Obfuscated ES (geoid tokens only)

In [ ]:
r = await client.post(
    f"/search/catalogs/{CATALOG_ID}/items-search",
    json={"collections": [COLLECTION_ID], "limit": 10},
)
data = r.json()
print(f"Search results: {len(data.get('features', []))}")
for f in data.get("features", []):
    # Obfuscated: only geoid visible, no owner/area attributes
    print(" ", f.get("id"), "properties:", list(f.get("properties", {}).keys()))